# Module 8: Capstone — Decision-Memo System

**"Brief in, leadership memo out."**

Combines all four patterns into one complete pipeline.

![Decision-Memo System: Planner + Researcher + Analyzers + Critic → Program Revisor → Decision Memo](./architecture.png)

**Patterns combined:**
- **P2 Parallel heads**: Planner, Researcher, Analyzer A, Analyzer B run simultaneously on the brief
- **P3 Critic-Refiner**: Program Revisor drafts → Critic approves or requests revision → loop
- **P5 Agent-as-Tool**: Orchestrator delegates to each specialist as a callable `@tool`
- **P1 Sequential synthesis**: Program Revisor synthesizes all inputs in sequence

**Agents:** Planner · Researcher · Analyzer (×2) · Program Revisor · Critic

**Prerequisites:** Modules 1–7. This module reuses tools from Module 2.

## Components in This Module

| Component | Pattern | What it does |
|-----------|---------|-------------|
| `parallel_heads` | P2 Fork-Join | Planner + Researcher + Analyzer A + Analyzer B run simultaneously |
| `program_revisor` | P3 Critic-Refiner | Program Revisor drafts → Critic approves or requests revision |
| `orchestrator` | P5 Agent-as-Tool | Coordinates both tools; LLM decides call sequence |

> **The `@tool` docstring IS the routing logic.** The orchestrator reads it to decide when and with what arguments to call each specialist.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1, Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2, Claude Haiku 4.5 (faster, lower cost):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3, Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
print("✅ Setup complete!")

In [ ]:
import sys, os, time, asyncio, json
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02-single-agent"))

import nest_asyncio
nest_asyncio.apply()

from strands import Agent, tool
from strands.multiagent import GraphBuilder
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

---

## Part 1: System Prompts

All four specialists share one model instance (passed at Agent creation). Narrow, focused prompts: each agent does exactly one job.

In [ ]:
PLANNER_PROMPT = (
    "You are a decision planner. Analyze the brief and produce a structured analysis plan: "
    "key questions to answer, what data is needed, what criteria matter for choosing between options. "
    "100 words max. Return structured plan only."
)

RESEARCHER_PROMPT = (
    "You are a market research specialist. Use your tools to gather relevant data. "
    "Return structured findings: data only, no recommendations."
)

ANALYZER_PROMPT = (
    "You are a business strategy analyst. Evaluate the ONE option you are given: "
    "strengths, weaknesses, complexity (Low/Med/High), top 2 risks with mitigations, verdict. "
    "100 words max."
)

PROGRAM_REVISOR_PROMPT = (
    "You are an executive memo writer. Synthesize the plan and analyses into a COMPLETE leadership memo:\n"
    "## Recommendation (one sentence: which option and why)\n"
    "## Options at a Glance (table comparing A, B, C on Complexity/Risk/Verdict)\n"
    "## Top 3 Risks with specific mitigations\n"
    "## Success Metrics (at least 2 KPIs with numeric targets)\n"
    "## Decision Required (owner, deadline, who approves)\n"
    "If you receive revision feedback, address ALL flagged criteria. Under 400 words."
)

CRITIC_PROMPT = (
    "You are a quality critic. Check ONLY these 5 criteria:\n"
    "1. ## Recommendation with a clear option choice (A, B, or C)\n"
    "2. ## Options at a Glance table comparing A, B, C\n"
    "3. ## Top 3 Risks with at least 3 risks each with a mitigation\n"
    "4. ## Success Metrics with at least 2 KPIs with numeric targets\n"
    "5. ## Decision Required with owner AND deadline\n"
    "Respond: APPROVED or REVISION NEEDED: [criteria numbers missing]"
)

ORCHESTRATOR_PROMPT = (
    "You are the Decision-Memo System orchestrator. Execute:\n"
    "1. Call parallel_heads with the decision brief — runs Planner, Researcher, and Analyzers simultaneously.\n"
    "2. Call program_revisor with the brief and all parallel findings — produces the quality-reviewed memo.\n"
    "Execute both steps in order."
)

---

## Part 2: Build the Two Specialist Tools

### Tool 1: `parallel_heads` (P2 Fork-Join)
Planner, Researcher, and two Analyzers run simultaneously — all receive the same brief and process it independently.

### Tool 2: `program_revisor` (P3 Critic-Refiner)
Program Revisor drafts the memo → Critic evaluates → loop until APPROVED.

In [ ]:
@tool
def parallel_heads(brief: str) -> str:
    '''Run Planner, Researcher, Analyzer A, and Analyzer B simultaneously on the decision brief.

    Returns combined output from all four specialists.

    Args:
        brief: The full decision brief
    '''
    planner   = Agent(system_prompt=PLANNER_PROMPT,   callback_handler=None)
    researcher = Agent(
        tools=[get_company_data, get_market_benchmarks, get_competitor_data],
        system_prompt=RESEARCHER_PROMPT, callback_handler=None,
    )
    analyzer_a = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)
    analyzer_b = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)
    analyzer_c = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)

    async def fork():
        return await asyncio.gather(
            planner.invoke_async(brief),
            researcher.invoke_async(brief),
            analyzer_a.invoke_async(f"Option A: Exclusive Premium ($19.99/mo, invite-only top 10%)\nBrief: {brief}"),
            analyzer_b.invoke_async(f"Option B: Gradual Rollout ($14.99/mo, 5% A/B pilot)\nBrief: {brief}"),
            analyzer_c.invoke_async(f"Option C: Full Launch ($12.99/mo, open to all, 30-day trial)\nBrief: {brief}"),
        )

    plan_out, research_out, a_out, b_out, c_out = asyncio.run(fork())
    return (
        f"PLAN:\n{plan_out}\n\n"
        f"RESEARCH:\n{research_out}\n\n"
        f"OPTION A:\n{a_out}\n\n"
        f"OPTION B:\n{b_out}\n\n"
        f"OPTION C:\n{c_out}"
    )

In [ ]:
@tool
def program_revisor(brief: str, parallel_findings: str) -> str:
    '''Synthesize all parallel findings into a leadership memo, with Critic quality review.

    Program Revisor drafts the memo. Critic evaluates it.
    If REVISION NEEDED, Program Revisor revises with feedback. Loops until APPROVED.

    Args:
        brief: The original decision brief
        parallel_findings: Combined output from parallel_heads (plan + research + analyses)
    '''
    revisor = Agent(name="program_revisor", system_prompt=PROGRAM_REVISOR_PROMPT, callback_handler=None)
    critic  = Agent(name="critic",          system_prompt=CRITIC_PROMPT,          callback_handler=None)

    def needs_revision(state):
        r = state.results.get("critic")
        return bool(r) and "revision needed" in str(r.result).lower()

    builder = GraphBuilder()
    builder.add_node(revisor, "program_revisor")
    builder.add_node(critic,  "critic")
    builder.set_entry_point("program_revisor")
    builder.add_edge("program_revisor", "critic")
    builder.add_edge("critic", "program_revisor", condition=needs_revision)
    builder.set_max_node_executions(6)
    builder.set_execution_timeout(180)
    builder.reset_on_revisit(True)

    result = builder.build()(
        f"Brief:\n{brief}\n\nParallel findings (plan + research + analyses):\n{parallel_findings}"
    )
    for node in reversed(result.execution_order):
        if node.node_id == "program_revisor":
            return str(node.result)
    return str(result)

---

## Part 3: The Orchestrator and Full Brief

In [ ]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Company: NovaCart (2M active users, mid-size e-commerce)
Decision owners: VP Product + CFO approval required

Options:
  Option A: Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B: Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C: Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

orchestrator = Agent(
    tools=[parallel_heads, program_revisor],
    system_prompt=ORCHESTRATOR_PROMPT,
)

print("Running Decision-Memo System...")
print("Step 1: parallel_heads (Planner + Researcher + Analyzers A/B/C)")
print("Step 2: program_revisor (Program Revisor ↔ Critic loop)")
t0 = time.time()
result = orchestrator(DECISION_BRIEF)
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")

---

## Part 4: Inspect the Pipeline

In [ ]:
print("=== PIPELINE EXECUTION ===")
call_n = 0
for msg in orchestrator.messages:
    for block in msg.get("content", []):
        if "toolUse" in block:
            call_n += 1
            tu = block["toolUse"]
            inp_keys = list(tu.get("input", {}).keys())
            print(f"  {call_n}. {tu['name']}({', '.join(inp_keys)})")

print()
print("Tool 1 (parallel_heads): Planner + Researcher + Analyzers A/B/C in parallel")
print("Tool 2 (program_revisor): Program Revisor ↔ Critic quality loop")

In [ ]:
# Token usage across the full pipeline
summary = result.metrics.get_summary()
usage = summary.get("accumulated_usage", {})

print(f"{'Metric':<25} {'Value':>10}")
print("-" * 37)
print(f"{'Input tokens':<25} {usage.get('inputTokens', 0):>10,}")
print(f"{'Output tokens':<25} {usage.get('outputTokens', 0):>10,}")
print(f"{'Total tokens':<25} {usage.get('totalTokens', 0):>10,}")
print(f"{'Orchestrator cycles':<25} {summary.get('total_cycles', 'n/a'):>10}")
print()

tool_stats = summary.get("tool_usage", {})
if tool_stats:
    print("Per-tool timing:")
    for name, data in tool_stats.items():
        s = data.get("execution_stats", {})
        print(f"  {name}: calls={s.get('call_count',0)} | avg_time={round(s.get('average_time',0),1)}s")

---

## What You Built

| Pattern | Component | Strands API |
|---------|-----------|-------------|
| P2 Parallel heads | Planner + Researcher + Analyzers A/B/C simultaneously | `asyncio.gather` + `invoke_async` |
| P3 Critic-Refiner | Program Revisor ↔ Critic quality loop | `GraphBuilder` + cycle edge |
| P5 Agent-as-Tool | Orchestrator delegates to both tools | `@tool` wrapping sub-pipelines |
| P1 Sequential | Program Revisor synthesizes after all parallel heads complete | Implicit: tool 2 runs after tool 1 |

**Decision Brief → Leadership Memo in ~60 seconds:**
- 1 orchestrator (P5) calling 2 tools
- 5 parallel agents (P2): Planner + Researcher + Analyzer×3
- 1 Program Revisor + 1 Critic in quality loop (P3)